In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)

In [2]:
def run_logreg_groupcv(X, y, groups, name, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)

    rows = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="liblinear",
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            ))
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        y_prob = pipe.predict_proba(X_test)[:, 1]

        rows.append({
            "feature_set": name,
            "fold": fold,
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_prob)
        })

    return pd.DataFrame(rows)

In [14]:
import pandas as pd
from pathlib import Path

FEATURE_DIR = Path("../data/features_participant")

X_pupil_only = pd.read_csv(FEATURE_DIR / "X_pupil_only.csv")
X_gaze_only  = pd.read_csv(FEATURE_DIR / "X_gaze_only.csv")
X_combined   = pd.read_csv(FEATURE_DIR / "X_combined.csv")
y            = pd.read_csv(FEATURE_DIR / "y.csv")["label"]

# Load participant IDs from your master participant-level feature file
ids_df = pd.read_csv("../data/X_pupil_participant.csv", usecols=["participant"])
groups = ids_df["participant"].to_numpy()

# sanity check alignment
assert len(groups) == len(y) == len(X_pupil_only) == len(X_gaze_only) == len(X_combined)

print("OK:", len(groups), "participants")

OK: 57 participants


In [16]:
results_pupil = run_logreg_groupcv(
    X_pupil_only, y, groups, name="pupil_only"
)

results_gaze = run_logreg_groupcv(
    X_gaze_only, y, groups, name="gaze_only"
)

results_combined = run_logreg_groupcv(
    X_combined, y, groups, name="combined"
)

results_all = pd.concat(
    [results_pupil, results_gaze, results_combined],
    ignore_index=True
)

In [17]:
summary = (
    results_all
    .groupby("feature_set")
    .agg(
        acc_mean=("accuracy", "mean"),
        acc_sd=("accuracy", "std"),
        bal_acc_mean=("balanced_accuracy", "mean"),
        bal_acc_sd=("balanced_accuracy", "std"),
        auc_mean=("roc_auc", "mean"),
        auc_sd=("roc_auc", "std"),
    )
    .round(3)
    .reset_index()
)

summary

,feature_set,acc_mean,acc_sd,bal_acc_mean,bal_acc_sd,auc_mean,auc_sd
0,combined,0.773,0.097,0.767,0.088,0.834,0.048
1,gaze_only,0.703,0.093,0.713,0.086,0.820,0.018
2,pupil_only,0.759,0.184,0.756,0.183,0.791,0.152


“Across group-wise cross-validation, the combined gaze+pupil feature set achieved the best performance (accuracy 0.77 ± 0.10; ROC-AUC 0.83 ± 0.05), outperforming gaze-only and yielding more stable results than pupil-only.”